## Example 1: Static and environment credential resolvers

In [ ]:
import importlib.metadata
import os

from smithy_aws_core.identity import AWSCredentialsIdentity, EnvironmentCredentialsResolver, StaticCredentialsResolver

# Example 1: resolvers hand a client its AWS identity
assert importlib.metadata.version("smithy-aws-core") == "0.12.0"
static = StaticCredentialsResolver(AWSCredentialsIdentity(access_key_id="AK", secret_access_key="SK"))
assert (await static.get_identity(properties={})).access_key_id == "AK"
os.environ.update(AWS_ACCESS_KEY_ID="AKENV", AWS_SECRET_ACCESS_KEY="SKENV")
env_identity = await EnvironmentCredentialsResolver().get_identity(properties={})
assert env_identity.access_key_id == "AKENV" and env_identity.secret_access_key == "SKENV"
print("Example 1 OK:", env_identity.access_key_id)

## Example 2: A chain tries resolvers in order

In [ ]:
import os

from smithy_aws_core.identity import AWSCredentialsIdentity, EnvironmentCredentialsResolver, IdentityChain, StaticCredentialsResolver

# Example 2: the environment wins when it has credentials, otherwise the next resolver answers
fallback = StaticCredentialsResolver(AWSCredentialsIdentity(access_key_id="AKFALLBACK", secret_access_key="SK"))
chain = IdentityChain([EnvironmentCredentialsResolver(), fallback])
os.environ.pop("AWS_ACCESS_KEY_ID", None); os.environ.pop("AWS_SECRET_ACCESS_KEY", None)
assert (await chain.get_identity(properties={})).access_key_id == "AKFALLBACK"
os.environ.update(AWS_ACCESS_KEY_ID="AKENV", AWS_SECRET_ACCESS_KEY="SKENV")
assert (await IdentityChain([EnvironmentCredentialsResolver(), fallback]).get_identity(properties={})).access_key_id == "AKENV"
print("Example 2 OK")

## Example 3: Parse a shared AWS config file

In [ ]:
import os
import tempfile

from smithy_aws_core.config.file_parser import FileType, parse_config_file, standardize
from smithy_aws_core.config.filesystem import DefaultFileSystem

# Example 3: ~/.aws/config syntax: [default] and [profile x] sections, nested service settings
path = os.path.join(tempfile.mkdtemp(), "config")
with open(path, "w") as f:
    f.write("[default]\nregion = us-east-1\n\n[profile gpu]\nregion = us-west-2\nmax_attempts = 5\n")
raw = parse_config_file(path, DefaultFileSystem())
raw = await raw if hasattr(raw, "__await__") else raw
config = standardize(raw, FileType.CONFIG)
assert config.profiles["default"].properties["region"] == "us-east-1"
assert config.profiles["gpu"].properties == {"region": "us-west-2", "max_attempts": "5"}, config.profiles["gpu"]
print("Example 3 OK:", sorted(config.profiles))

## Example 4: Credentials that expire

In [ ]:
import datetime

from smithy_aws_core.identity import AWSCredentialsIdentity

# Example 4: temporary credentials carry an expiry and report when they are past it
now = datetime.datetime.now(datetime.timezone.utc)
fresh = AWSCredentialsIdentity(access_key_id="ASIA1", secret_access_key="SK", session_token="t", expiration=now + datetime.timedelta(hours=1))
stale = AWSCredentialsIdentity(access_key_id="ASIA2", secret_access_key="SK", session_token="t", expiration=now - datetime.timedelta(minutes=1))
forever = AWSCredentialsIdentity(access_key_id="AKIA", secret_access_key="SK")
assert not fresh.is_expired and stale.is_expired and not forever.is_expired
print("Example 4 OK")

## Example 5: Signing goes through the SigV4 auth scheme

In [ ]:
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import EnvironmentCredentialsResolver

# Example 5: the auth scheme a generated client registers: its id, how it signs and how it finds an identity
scheme = SigV4AuthScheme(service="bedrock")
assert str(scheme.scheme_id) == "aws.auth#sigv4", scheme.scheme_id
assert callable(scheme.signer) and callable(scheme.identity_resolver), (scheme.signer, scheme.identity_resolver)
assert EnvironmentCredentialsResolver is not None
print("Example 5 OK:", scheme.scheme_id)